In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nltk.sentiment import SentimentIntensityAnalyzer

import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...


True

In [2]:
news_df = pd.read_csv("../data/raw/raw_analyst_ratings.csv")

news_df.head()

,Unnamed: 0,headline,url,publisher,date,stock
0,0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 10:30:54-04:00,A
1,1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 10:45:20-04:00,A
2,2,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 04:30:07-04:00,A
3,3,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 12:45:06-04:00,A
4,4,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 11:38:59-04:00,A


In [3]:
news_df = news_df[news_df["stock"] == "AAPL"]

In [ ]:
news_df["date"] = pd.to_datetime(news_df["date"])

news_df["date_only"] = news_df["date"].dt.datenews_df["date"] = pd.to_datetime(
    news_df["date"],
    format="mixed",
    errors="coerce"
)

news_df["date_only"] = news_df["date"].dt.date

ValueError: time data "2020-06-02 00:00:00" doesn't match format "%Y-%m-%d %H:%M:%S%z". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
sia = SentimentIntensityAnalyzer()

news_df["sentiment"] = news_df["headline"].apply(
    lambda x: sia.polarity_scores(x)["compound"]
)

In [ ]:
news_df[["headline", "sentiment"]].head()

In [ ]:
daily_sentiment = news_df.groupby("date_only")["sentiment"].mean()

daily_sentiment = daily_sentiment.reset_index()

daily_sentiment.head()

In [ ]:
stock_df = pd.read_csv("../data/raw/AAPL.csv")

stock_df.head()

In [ ]:
stock_df["Date"] = pd.to_datetime(stock_df["Date"])

stock_df["date_only"] = stock_df["Date"].dt.date

In [ ]:
stock_df["daily_return"] = stock_df["Close"].pct_change() * 100

In [ ]:
merged_df = pd.merge(
    daily_sentiment,
    stock_df,
    on="date_only"
)

merged_df.head()

In [ ]:
correlation = merged_df["sentiment"].corr(
    merged_df["daily_return"]
)

print("Correlation:", correlation)

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    merged_df["sentiment"],
    merged_df["daily_return"]
)

plt.xlabel("Average Daily Sentiment")
plt.ylabel("Daily Stock Return")

plt.title("Sentiment vs Daily Stock Return")

plt.show()

In [ ]:
def classify_sentiment(score):
    if score > 0.05:
        return "Positive"
    elif score < -0.05:
        return "Negative"
    else:
        return "Neutral"

merged_df["sentiment_category"] = merged_df["sentiment"].apply(classify_sentiment)

In [ ]:
category_returns = merged_df.groupby(
    "sentiment_category"
)["daily_return"].mean()

category_returns.plot(kind="bar")

plt.title("Average Daily Return by Sentiment Category")

plt.ylabel("Average Return")

plt.show()